# Week 7: Spatial regression models

We continue today looking at regression models, thinking about how they apply in a spatial context. Here, we are now looking at how predictor and confounding variables of a certain geography impact some outcome variable. For example, what happens if our data comes from aggregated spatial markers? 

## 7.1 Loading Spatial Census Data

While the mechanics of a spatial regression might look similar to other simple and multiple regressions, they are different in that geography itself is a feature in the model. 

For example, think of the case in which we want to understand how home prices vary based on being closer or further from set features like the beach, or a highway, or public transit. Simply the physical location of a home impacts the price potentially as much as its other features like number of bedrooms or bathrooms. 

It's also possible that geography is something we want to account for; it is something that explains some variance or error in order model that we can't understand from the features we have. 

In [ ]:
#Import your census key from Week 4
from key import CENSUS_KEY
import pandas as pd
import numpy as np 
import geopandas as gpd
from census import Census
import statsmodels.formula.api as smf
from us import states
import matplotlib.pyplot as plt

# Initialize the API
c = Census(CENSUS_KEY)


In [ ]:
# Let's pull some data from the census again
census_variables = {'B01001_026E': 'Female',
                     'B03002_003E': 'White',
                     'B03002_004E': 'Black',
                     'B03001_003E': 'Hispanic',
                     'B03002_006E': 'Asian',
                     'B06011_001E': 'MedIncome',
                     'B01003_001E': 'TotalPop',
                     'B10059_002E': 'Poverty',
                     'B15003_022E': 'Bach',
                     'B06009_006E': 'GradPro',
                     'B07001_005E': 'Age2024',
                     'B07001_006E': 'Age2529',
                     'B07001_007E': 'Age3034',
                     'B08006_003E': 'DroveAlone',
                     'B08006_008E': 'Transit',
                     'B08006_001E': 'Commuters',
                     'B25106_024E': 'Renter',
                     'B25002_003E': 'Vacant',
                     'B25002_001E': 'TotalUnits',
                     'B25024_002E': 'Singleh',
                     'B25024_001E': 'TotalHouse',
                     'B25097_001E': 'MedHvalue'}

chi_tracts = c.acs5.get( # start here each time
    list(census_variables.keys()), # specify the variables we want, # specify the variables we want
    {'for': 'tract:*', # specify the geometry resolution we want
     'in': 'state:17 county:031'}, # specify the geometry bounds. In this case, the state of IL, Cook County
    year=2020, # specify the year
)
chi_tracts = pd.DataFrame(chi_tracts).rename(columns=census_variables)
chi_tracts['GEOID'] = chi_tracts['state'] + chi_tracts['county'] + chi_tracts['tract']


In [ ]:
# Let's add our young ages together
chi_tracts['Age2029'] = chi_tracts['Age2024'] + chi_tracts['Age2529']

# Let's get rid of values < 0, these are errors
chi_tracts[chi_tracts[census_variables.values()] < 0] = np.nan

# Finally, let's create a few percentage columns
def create_percent_col(col_name, df, totalpop='TotalPop'):
    """
    Add a column that divdes column name values by total population.
    Takes col_name and the dataframe as inputs. totalpop col name optional
    Outputs the dataframe with the new column.
    """
    df[f"percent_{col_name}"] = 100 * df[col_name] / df[totalpop]
    return df
    

columns = ['White', 'Black', 'Hispanic', 'Asian', 'Poverty', 'Bach', 'GradPro', \
           'Age2029','Age3034', 'DroveAlone', 'Transit', 'Commuters', 'Renter']

for col_name in columns:
    chi_tracts = create_percent_col(col_name, chi_tracts)

chi_tracts.head()

In [ ]:
# Lets add the geometry information
year=2020
state_fips='17'
url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{state_fips}_tract.zip"
tracts = gpd.read_file(url)
tracts = tracts.to_crs(epsg=4326)

In [ ]:
# And merge them on the GEOID 
chi_tracts = pd.merge(left=chi_tracts, right=tracts, on='GEOID', how='left')
chi_tracts = gpd.GeoDataFrame(chi_tracts)
chi_tracts.crs = "EPSG:4326"
chi_tracts.head()

In [ ]:
# We can plot one of our columns, for example MedIncome
ax = chi_tracts.plot(column='MedIncome')
ax.axis("off") # Remove the lat/lon axes labels


Let's build a regression model for median income

In [ ]:
model = smf.ols(formula='MedIncome ~ percent_White + percent_White + percent_Black + \
                percent_Hispanic + percent_Asian + percent_Poverty + percent_Bach + \
                percent_GradPro + percent_Age2029', data=chi_tracts).fit()

In [ ]:
model.summary()

## 7.2 Spatial Weights

There may be relationships between our census tracts that mean each data point (tract) is not independent. Because we don't necessarily live by the boundaries of census geometries, there may be "spillover effects" from one to its neighbor. Consider the map of Chicago above, there seem to be spatially dependent factors that dictact how high or low median income is -- maybe moving closer to the city? Maybe living north of the city? 

**Spatial weights** take the form of a matrix and tell us how our points (in this case tracts) are related to eachother. There are a few types:
* Rook contiguity
* Queen contiguity
* k-nearest neighbors
* distance band

In [ ]:
!pip install libpysal

### 7.2.1 Rook Contiguity

Like chess, in Rook contiguity, two census tracts must share an edge of their boundaries to be considered neighbors. Even though our tracts aren't usually squares, we can still understand this to mean: neighbors must share an edge. 

In [ ]:
from libpysal import io
from libpysal import weights

In [ ]:
# We can pick one of our tracts and set our GEOID column to be the index
geoid ="17031062600"

chi_tracts = chi_tracts.set_index('GEOID') 

In [ ]:
# Let's look at the spatial weights using Rook
w_rook = weights.Rook.from_dataframe(chi_tracts, use_index=True)

In [ ]:
w_rook[geoid]

### 7.2.2 Queen Contiguity

Similar the Rook, Queen contiguity takes its name from chess. However, the queen can move front/back, left/right, and diagonally. In our case, two tracts must now only share a point (or vertex) to be considered neighbors. 

In [ ]:
# Let's look at the spatial weights using Queen
w_queen = weights.Queen.from_dataframe(chi_tracts, use_index=True)

In [ ]:
w_queen[geoid]

In [ ]:
# How many neighbors does the the average tract have? 
print(w_queen.min_neighbors) # min
print(w_queen.max_neighbors) # max
print(w_queen.islands) # these are tracts with no neighbors, can cause issues down the line

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
chi_tracts.plot(ax=ax, facecolor="#666666", edgecolor="w", linewidth=0.5)

# plot some tract of interest in red
tract = chi_tracts.loc[[geoid]]
tract.plot(ax=ax, facecolor="#ff0000", edgecolor="w", linewidth=2)

# plot the neighbors in blue
neighbors = chi_tracts.loc[w_queen[geoid].keys()]
neighbors.plot(ax=ax, facecolor="#0033cc", edgecolor="w", linewidth=2)

# zoom to area of interest
xmin, ymin, xmax, ymax = neighbors.union_all().bounds
ax.axis("equal")
ax.set_xlim(xmin - .1, xmax + .1)  # +/- 100 meters
ax.set_ylim(ymin - .1, ymax + .1)

ax.set_title("Neighbors of tract {}".format(geoid))
_ = ax.axis("off")

### 7.2.3 k-nearest neighbors

This method looks for the _k_ nearest neighbors to each tract, regardless of if they physically share a boundary or not. 

In [ ]:
w_knn = weights.KNN.from_dataframe(chi_tracts, k=7)

In [ ]:
w_knn.neighbors[geoid]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
chi_tracts.plot(ax=ax, facecolor="#666666", edgecolor="w", linewidth=0.5)

# plot some tract of interest in red
tract = chi_tracts.loc[[geoid]]
tract.plot(ax=ax, facecolor="#ff0000", edgecolor="w", linewidth=2)

# plot the neighbors in blue
neighbors = chi_tracts.loc[w_knn.neighbors[geoid]]
neighbors.plot(ax=ax, facecolor="#0033cc", edgecolor="w", linewidth=2)

# zoom to area of interest
xmin, ymin, xmax, ymax = neighbors.union_all().bounds
ax.axis("equal")
ax.set_xlim(xmin - .1, xmax + .1)  # +/- 100 meters
ax.set_ylim(ymin - .1, ymax + .1)

ax.set_title("Neighbors of tract {}".format(geoid))
_ = ax.axis("off")

In [ ]:
## YOUR TURN 
## Compute the k-nearest neighbors for k = 4 and another for k = 10




In [ ]:
# Let's plot both
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 10))

# left plot (k=4)
chi_tracts.plot(ax=axes[0], facecolor="#666666", edgecolor="w", linewidth=0.5)
# plot some tract of interest in red
tract = chi_tracts.loc[[geoid]]
tract.plot(ax=axes[0], facecolor="#ff0000", edgecolor="w", linewidth=2)
# plot the neighbors in blue
neighbors = chi_tracts.loc[w_knn4.neighbors[geoid]]
neighbors.plot(ax=axes[0], facecolor="#0033cc", edgecolor="w", linewidth=2)
# zoom to area of interest
xmin, ymin, xmax, ymax = neighbors.union_all().bounds
axes[0].axis("equal")
axes[0].set_xlim(xmin - .1, xmax + .1)  # +/- 100 meters
axes[0].set_ylim(ymin - .1, ymax + .1)
_ = axes[0].axis("off")


#right plot (k=10)
chi_tracts.plot(ax=axes[1], facecolor="#666666", edgecolor="w", linewidth=0.5)
# plot some tract of interest in red
tract = chi_tracts.loc[[geoid]]
tract.plot(ax=axes[1], facecolor="#ff0000", edgecolor="w", linewidth=2)
# plot the neighbors in blue
neighbors = chi_tracts.loc[w_knn10.neighbors[geoid]]
neighbors.plot(ax=axes[1], facecolor="#0033cc", edgecolor="w", linewidth=2)
# zoom to area of interest
xmin, ymin, xmax, ymax = neighbors.union_all().bounds
axes[1].axis("equal")
axes[1].set_xlim(xmin - .1, xmax + .1)  # +/- 100 meters
axes[1].set_ylim(ymin - .1, ymax + .1)
_ = axes[1].axis("off")

### 7.2.4 Distance bands

Distance weights involve including neighbors if they are some distance away and can have continuous values, not just 0/1. We will not cover this in this class

### 7.2.5 Standarized Weights

Sometimes we want the weights to not be just 0/1 for each neighbor, but be standarized across each row. Typically we would like each row to sum to 1. 

In [ ]:
# Recall our queen neighbors
w_queen[geoid]

In [ ]:
w_queen.set_transform("R") # R = values of neighbors sum to 1 for each row (tract)

In [ ]:
w_queen[geoid]

# 7.3 Spatial Lag

Spatial lag gets to our question of are things close together similar? We use spatial weights to find what it means to be "close together" and we use spatial lag to tell us if they are similar or not. 

The standarization of weights is important because now, no matter how the weights are built, the spatial lag is the **average value of a tract's neighbors**. 

In [ ]:
# Let's look at our median income
# It's important that there are not any null values

chi_tracts_not_na = chi_tracts.dropna()

In [ ]:
# Let's get our queen weights and standardize 
w_queen = weights.Queen.from_dataframe(chi_tracts_not_na, use_index=True)
w_queen.set_transform("R")

In [ ]:
# Our column of interest is Median Income
y = chi_tracts_not_na['MedIncome']

# We can compute spatial lag
y_lag = weights.lag_spatial(w_queen, y)

In [ ]:
# Get the index location of our geoid
geoid_idx = y.index.get_loc(geoid)
geoid_idx

In [ ]:
# Compare the median income values of neighbors to the y_lag value
chi_tracts_not_na.loc[w_queen.neighbors[geoid]]

In [ ]:
print(chi_tracts_not_na.loc[w_queen.neighbors[geoid]]['MedIncome'].mean()) # mean of neighbors MedIncome
print(y_lag[geoid_idx]) #Spatial Lag of geoid MedIncome

In [ ]:
# is a tract's med income similar to those of its neighbors?
data_lag = pd.DataFrame(data={"MedIncome": y, "MedIncome_lag": y_lag}).astype(int)
data_lag

## 7.5 Adding Spatial Lag to a Model

We can add spatial lag to our regression model

In [ ]:
chi_tracts_not_na['spatial_lag'] = y_lag

In [ ]:
# Recall our previous model
model = smf.ols(formula='MedIncome ~ percent_White + percent_White + percent_Black + \
                percent_Hispanic + percent_Asian + percent_Poverty + percent_Bach + \
                percent_GradPro + percent_Age2029', data=chi_tracts_not_na).fit()
model.summary()

In [ ]:
model_lag = smf.ols(formula='MedIncome ~ spatial_lag + percent_White + percent_White + percent_Black + \
                percent_Hispanic + percent_Asian + percent_Poverty + percent_Bach + \
                percent_GradPro + percent_Age2029', data=chi_tracts_not_na).fit()
model_lag.summary()

In [ ]:
## YOUR TURN
## Experiment with different weight matrices to add spatial lag into a model 

